# Kenya Public Transport Demand Analysis & Forecasting

This project analyzes public transport statistics from five Kenyan counties — **Kisumu, Mombasa, Nairobi, Nakuru, and Uasin Gishu** — spanning 2013–2024. The goal is to understand transportation demand patterns, build forecasting models, and propose optimization strategies.

**Data Source:** Kenya National Bureau of Statistics (KNBS) Economic Surveys, with county-level estimates derived from Gross County Product weights, population data, and known transport patterns.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
print('All libraries imported successfully.')

## 1. Data Loading and Cleaning

### Task
Load each of the five county datasets into separate DataFrames, clean the multi-row headers, and address data quality issues. The CSVs have:
1. A title row (county name)
2. A column header row
3. A units row
4. Data rows

We need to skip the title and units rows, parse headers correctly, and convert all values to numeric types.

In [ ]:
import os

DATA_DIR = os.path.dirname(os.path.abspath('__file__'))

def clean_dataset(file_path, county_name):
    '''Load a county CSV, parse multi-row headers, and return a clean DataFrame.'''
    df_raw = pd.read_csv(file_path, header=None)

    # Row 0 = title, Row 1 = column headers, Row 2 = units, Row 3+ = data
    headers = df_raw.iloc[1].astype(str).str.strip()

    col_names = ['Year']
    for i in range(1, len(headers)):
        name = headers[i]
        if name in ('nan', ''):
            name = f'Unknown_{i}'
        col_names.append(name)

    df = df_raw.iloc[3:].copy()
    df.columns = col_names[:len(df.columns)]

    for col in df.columns:
        df[col] = df[col].astype(str).str.replace(',', '').str.strip().str.strip('"')
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['Year'])
    df['Year'] = df['Year'].astype(int)
    df['County'] = county_name

    rename_map = {}
    for col in df.columns:
        cl = col.lower().strip()
        if 'road transport' in cl:
            rename_map[col] = 'Road Transport'
        elif 'panelvan' in cl or 'pick-up' in cl:
            rename_map[col] = 'PanelVans, Pick-ups'
        elif cl == 'minibuses/matatu':
            rename_map[col] = 'MiniBuses/Matatu'
        elif 'total motor' in cl:
            rename_map[col] = 'Total Motor Cycles'
        elif 'matatus' in cl and '0-14' in cl:
            rename_map[col] = 'Matatus (0-14 seaters)'
        elif 'buses' in cl and '34' in cl:
            rename_map[col] = 'Buses (34+ seaters)'
        elif ('mini bus' in cl or 'mini bues' in cl) and '15-33' in cl:
            rename_map[col] = 'Mini Buses (15-33 seaters)'
        elif 'buses and coaches' in cl:
            rename_map[col] = 'Buses and Coaches'

    df = df.rename(columns=rename_map)

    keep_cols = ['Year', 'Road Transport', 'PanelVans, Pick-ups', 'MiniBuses/Matatu',
                 'Total Motor Cycles', 'Matatus (0-14 seaters)', 'Buses (34+ seaters)',
                 'Mini Buses (15-33 seaters)', 'Buses and Coaches', 'County']
    df = df[[c for c in keep_cols if c in df.columns]]
    return df.reset_index(drop=True)


datasets = {
    'Kisumu': 'Kisumu dataset.csv',
    'Mombasa': 'Mombasa Dataset.csv',
    'Nairobi': 'Nairobi dataset.csv',
    'Nakuru': 'Nakuru Dataset.csv',
    'Uasin Gishu': 'Uasin Gishu Dataset.csv',
}

county_dfs = {}
for county, filename in datasets.items():
    filepath = os.path.join(DATA_DIR, filename)
    county_dfs[county] = clean_dataset(filepath, county)
    print(f'{county}: {len(county_dfs[county])} rows loaded')

df_kisumu = county_dfs['Kisumu']
df_mombasa = county_dfs['Mombasa']
df_nairobi = county_dfs['Nairobi']
df_nakuru = county_dfs['Nakuru']
df_uasingishu = county_dfs['Uasin Gishu']

print(f'\nSample - Nairobi:')
print(df_nairobi.head())

## 2. Data Consolidation

Combine the five cleaned county DataFrames into a single unified DataFrame for comprehensive cross-county analysis and model training.

In [ ]:
df_combined = pd.concat(
    [df_kisumu, df_mombasa, df_nairobi, df_nakuru, df_uasingishu],
    ignore_index=True
)

print(f'Combined dataset: {df_combined.shape[0]} rows x {df_combined.shape[1]} columns')
print(f'Counties: {df_combined["County"].unique()}')
print(f'Years: {sorted(df_combined["Year"].unique())}')
print(f'\nData types:')
print(df_combined.dtypes)
print(f'\nMissing values:')
print(df_combined.isnull().sum())
print(f'\nDescriptive statistics:')
df_combined.describe()

## 3. Exploratory Data Analysis (EDA)

Perform in-depth exploratory analysis on the consolidated dataset:
- Visualize trends over time by county
- Examine correlations between transport categories
- Identify distributions and outliers

In [ ]:
print('First 10 rows of combined dataset:')
print(df_combined.head(10))
print(f'\nShape: {df_combined.shape}')
print(f'\nDescriptive statistics:')
print(df_combined.describe().round(0))

In [ ]:
plt.figure(figsize=(12, 6))
for county in df_combined['County'].unique():
    data = df_combined[df_combined['County'] == county]
    plt.plot(data['Year'], data['Road Transport'], marker='o', label=county, linewidth=2)
plt.title('Road Transport Volume Over Years by County (KSh Millions)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Road Transport (KSh Millions)')
plt.xticks(df_combined['Year'].unique(), rotation=45)
plt.legend(title='County')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
transport_categories = [
    'Total Motor Cycles',
    'Matatus (0-14 seaters)',
    'Buses (34+ seaters)',
    'Mini Buses (15-33 seaters)',
    'PanelVans, Pick-ups',
    'Buses and Coaches'
]

fig, axes = plt.subplots(3, 2, figsize=(16, 18))
axes = axes.flatten()

for i, category in enumerate(transport_categories):
    for county in df_combined['County'].unique():
        data = df_combined[df_combined['County'] == county]
        axes[i].plot(data['Year'], data[category], marker='o', label=county, linewidth=1.5)
    axes[i].set_title(f'{category} by County', fontsize=12)
    axes[i].set_xlabel('Year')
    axes[i].set_ylabel('Count')
    axes[i].legend(title='County', fontsize=8)
    axes[i].tick_params(axis='x', rotation=45)
    axes[i].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = df_combined.select_dtypes(include=['number']).columns
correlation_matrix = df_combined[numeric_cols].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, center=0, square=True)
plt.title('Correlation Matrix of Transportation Categories', fontsize=14)
plt.tight_layout()
plt.show()

print('\nKey correlations with Road Transport:')
corr_with_target = correlation_matrix['Road Transport'].drop('Road Transport').sort_values(ascending=False)
for col, val in corr_with_target.items():
    print(f'  {col:.<40s} {val:+.3f}')

In [ ]:
box_categories = ['Road Transport', 'Total Motor Cycles', 'Matatus (0-14 seaters)',
                  'Buses (34+ seaters)', 'Mini Buses (15-33 seaters)', 'PanelVans, Pick-ups']

fig, axes = plt.subplots(3, 2, figsize=(16, 16))
axes = axes.flatten()

for i, category in enumerate(box_categories):
    sns.boxplot(data=df_combined, x='County', y=category, ax=axes[i], palette='Set2')
    axes[i].set_title(f'{category} Distribution by County', fontsize=11)
    axes[i].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

## 4. Feature Engineering for Forecasting

Create features to enhance model performance:
- **Lag features**: Previous year's values (lag-1 and lag-2) for each transport category
- **Growth rates**: Year-over-year percentage change
- **County encoding**: Numeric representation of counties
- **Time trend**: Linear year index for capturing secular trends

In [ ]:
df_combined = df_combined.sort_values(['County', 'Year']).reset_index(drop=True)

demand_columns = [
    'Road Transport', 'PanelVans, Pick-ups', 'MiniBuses/Matatu',
    'Total Motor Cycles', 'Matatus (0-14 seaters)', 'Buses (34+ seaters)',
    'Mini Buses (15-33 seaters)', 'Buses and Coaches'
]

# Create lag-1 and lag-2 features grouped by County
for col in demand_columns:
    df_combined[f'{col}_lag1'] = df_combined.groupby('County')[col].shift(1)
    df_combined[f'{col}_lag2'] = df_combined.groupby('County')[col].shift(2)

# Year-over-year growth rate for Road Transport
df_combined['Road_Transport_growth'] = df_combined.groupby('County')['Road Transport'].pct_change()

# Time trend feature
df_combined['Year_index'] = df_combined['Year'] - df_combined['Year'].min()

# Label encode County
le = LabelEncoder()
df_combined['County_encoded'] = le.fit_transform(df_combined['County'])

print(f'Dataset after feature engineering: {df_combined.shape}')
lag_cols = [c for c in df_combined.columns if 'lag' in c or 'growth' in c or 'index' in c or 'encoded' in c]
print(f'New features ({len(lag_cols)}): {lag_cols}')
print(f'\nRows with NaN (from lagging): {df_combined.isnull().any(axis=1).sum()}')
print(f'Usable rows after dropping NaN: {df_combined.dropna().shape[0]}')

## 5. Model Selection and Training

Train multiple forecasting models to predict **Road Transport** demand:
- **Random Forest Regressor**: Handles non-linear relationships, robust to outliers
- **Gradient Boosting Regressor**: Strong sequential learner, often best for tabular data
- **Linear Regression**: Simple baseline for comparison

**Train/Test Split**: Train on 2013–2021, test on 2022–2024 (time-based split to avoid data leakage).

In [ ]:
target = 'Road Transport'

feature_cols = ['Year', 'Year_index', 'County_encoded', 'Road_Transport_growth']
feature_cols += [c for c in df_combined.columns if '_lag1' in c or '_lag2' in c]

df_model = df_combined.dropna(subset=feature_cols + [target]).copy()

X = df_model[feature_cols]
y = df_model[target]

train_mask = df_model['Year'] <= 2021
test_mask = df_model['Year'] >= 2022

X_train, X_test = X[train_mask], X[test_mask]
y_train, y_test = y[train_mask], y[test_mask]

print(f'Training set: {X_train.shape[0]} samples ({df_model[train_mask]["Year"].min()}-{df_model[train_mask]["Year"].max()})')
print(f'Test set:     {X_test.shape[0]} samples ({df_model[test_mask]["Year"].min()}-{df_model[test_mask]["Year"].max()})')
print(f'Features:     {X_train.shape[1]}')
print(f'\nFeature list:')
for f in feature_cols:
    print(f'  - {f}')

models = {
    'Random Forest': RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42),
    'Gradient Boosting': GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                     learning_rate=0.1, random_state=42),
    'Linear Regression': LinearRegression()
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results[name] = {'model': model, 'predictions': y_pred}
    print(f'\n{name} trained successfully.')

## 6. Model Evaluation

Evaluate model performance using:
- **MAE** (Mean Absolute Error): Average prediction error in KSh
- **RMSE** (Root Mean Squared Error): Penalizes large errors more
- **R² Score**: Proportion of variance explained (1.0 = perfect)
- **MAPE** (Mean Absolute Percentage Error): Error as percentage of actual values

In [ ]:
print('=' * 80)
print(f'{"Model":<25} {"MAE":>12} {"RMSE":>12} {"R2":>8} {"MAPE":>8}')
print('=' * 80)

best_model_name = None
best_r2 = -999

for name, res in results.items():
    y_pred = res['predictions']
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    mape = np.mean(np.abs((y_test - y_pred) / y_test)) * 100

    res['mae'] = mae
    res['rmse'] = rmse
    res['r2'] = r2
    res['mape'] = mape

    print(f'{name:<25} {mae:>12,.0f} {rmse:>12,.0f} {r2:>8.3f} {mape:>7.1f}%')

    if r2 > best_r2:
        best_r2 = r2
        best_model_name = name

print('=' * 80)
print(f'\nBest model: {best_model_name} (R2 = {best_r2:.3f})')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, (name, res) in enumerate(results.items()):
    y_pred = res['predictions']
    ax = axes[i]
    ax.scatter(y_test, y_pred, alpha=0.7, edgecolors='k', linewidth=0.5, s=80)
    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')
    ax.set_xlabel('Actual Road Transport (KSh M)', fontsize=11)
    ax.set_ylabel('Predicted Road Transport (KSh M)', fontsize=11)
    ax.set_title(f'{name}\nR2={res["r2"]:.3f}, MAPE={res["mape"]:.1f}%', fontsize=12)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Detailed predictions by county
best_preds = results[best_model_name]['predictions']
pred_df = df_model[test_mask][['Year', 'County', 'Road Transport']].copy()
pred_df['Predicted'] = best_preds.astype(int)
pred_df['Error'] = pred_df['Road Transport'] - pred_df['Predicted']
pred_df['Error_%'] = (pred_df['Error'] / pred_df['Road Transport'] * 100).round(1)
print(f'\nDetailed Predictions ({best_model_name}):')
print(pred_df.to_string(index=False))

In [ ]:
rf_model = results['Random Forest']['model']
importances = pd.Series(rf_model.feature_importances_, index=feature_cols)
importances = importances.sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importance (Random Forest)', fontsize=14)
plt.xlabel('Importance')
plt.tight_layout()
plt.show()

print('\nTop 5 most important features:')
for feat, imp in importances.sort_values(ascending=False).head(5).items():
    print(f'  {feat:.<50s} {imp:.4f}')

## 7. Forecasting Future Demand (2025–2027)

Use the best-performing model to forecast road transport demand for 2025–2027. We generate future features by projecting lag values from the most recent observed data.

In [ ]:
best_model = results[best_model_name]['model']
forecast_years = [2025, 2026, 2027]
forecasts = []

for county in df_combined['County'].unique():
    county_data = df_combined[df_combined['County'] == county].sort_values('Year').copy()
    county_enc = le.transform([county])[0]

    for year in forecast_years:
        latest = county_data.iloc[-1]
        second_latest = county_data.iloc[-2] if len(county_data) > 1 else latest

        features = {
            'Year': year,
            'Year_index': year - df_combined['Year'].min(),
            'County_encoded': county_enc,
            'Road_Transport_growth': (
                (latest['Road Transport'] - second_latest['Road Transport'])
                / second_latest['Road Transport']
            ) if second_latest['Road Transport'] != 0 else 0,
        }

        for col in demand_columns:
            features[f'{col}_lag1'] = latest[col]
            features[f'{col}_lag2'] = second_latest[col]

        X_forecast = pd.DataFrame([features])[feature_cols]
        prediction = best_model.predict(X_forecast)[0]

        forecasts.append({
            'Year': year,
            'County': county,
            'Predicted Road Transport (KSh M)': int(round(prediction))
        })

        # Add predicted row for next year's lag features
        new_row = latest.copy()
        new_row['Year'] = year
        new_row['Road Transport'] = prediction
        county_data = pd.concat([county_data, pd.DataFrame([new_row])], ignore_index=True)

forecast_df = pd.DataFrame(forecasts)
print('Road Transport Demand Forecast (2025-2027):')
print('=' * 60)
pivot = forecast_df.pivot(index='County', columns='Year', values='Predicted Road Transport (KSh M)')
print(pivot.to_string())

# Plot forecasts alongside historical data
plt.figure(figsize=(14, 7))
for county in df_combined['County'].unique():
    hist = df_combined[df_combined['County'] == county]
    fore = forecast_df[forecast_df['County'] == county]
    plt.plot(hist['Year'], hist['Road Transport'], marker='o', linewidth=2, label=f'{county} (actual)')
    plt.plot(fore['Year'], fore['Predicted Road Transport (KSh M)'], marker='s',
             linewidth=2, linestyle='--', alpha=0.7)

plt.axvline(x=2024.5, color='gray', linestyle=':', alpha=0.5, label='Forecast boundary')
plt.title('Road Transport Demand: Historical + Forecast (2025-2027)', fontsize=14)
plt.xlabel('Year')
plt.ylabel('Road Transport (KSh Millions)')
plt.legend(title='County', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Distributed System Architecture Design

### Subtask:
Outline a conceptual architecture for a distributed system capable of handling data ingestion, preprocessing, model training, real-time inference, and serving the transportation demand forecasts at scale.

#### Conceptual Architecture Outline

1.  **Data Ingestion Layer**:
    *   **Purpose**: To collect and ingest raw data from various heterogeneous sources.
    *   **Sources**: Real-time traffic sensors, public transport APIs, historical transportation datasets (like the county CSVs), weather APIs, event calendars, etc.
    *   **Technologies**: We would primarily use **Apache Kafka** as a high-throughput, fault-tolerant distributed streaming platform. Kafka topics would be set up for different data streams (e.g., `traffic_sensor_data`, `public_transport_api_data`, `historical_data_batches`).
    *   **Mechanism**: Data producers would push real-time data to Kafka topics. Batch data (like historical CSVs) would be ingested using batch connectors or scripts that write to Kafka.

2.  **Data Preprocessing and Storage Layer**:
    *   **Purpose**: To clean, transform, enrich, and store the ingested raw data for further processing and analysis.
    *   **Processing Technologies**: **Apache Spark** (Spark Streaming for real-time, Spark Batch for historical/large datasets) would be used for distributed data processing. Spark would consume data from Kafka, perform necessary cleaning (handling missing values, data type conversions, outlier detection), transformations (feature engineering, aggregation), and enrichment (joining with external datasets like weather).
    *   **Storage Solutions**:
        *   **Data Lake (e.g., HDFS or Cloud Storage like AWS S3/GCP Cloud Storage)**: For storing raw, semi-processed, and fully processed data in various formats (Parquet, ORC, CSV) to maintain a single source of truth and enable flexible querying.
        *   **Distributed Database (e.g., Apache Cassandra, MongoDB, or a cloud-native NoSQL DB)**: For storing processed features ready for model training and serving, especially for high-speed access patterns.

3.  **Model Training Layer**:
    *   **Purpose**: To manage, train, validate, and retrain forecasting models efficiently and scalably.
    *   **Model Management**: A **MLflow** server or similar platform for tracking experiments, managing model versions, and registering trained models.
    *   **Training Infrastructure**:
        *   **Distributed Machine Learning Frameworks**: **Apache Spark MLlib** for scalable training of machine learning models on large datasets, or integrating with specialized libraries like **XGBoost** or **TensorFlow/PyTorch** running on distributed clusters.
        *   **Cloud ML Services**: (e.g., AWS SageMaker, GCP AI Platform, Azure Machine Learning) could be utilized for managed training environments, auto-scaling compute resources, and hyperparameter tuning.
    *   **Retraining**: Models would be periodically retrained (e.g., monthly, quarterly, or on data drift detection) using fresh, preprocessed data from the Data Lake. New models would be validated against holdout datasets and potentially A/B tested.

4.  **Real-time Inference and Serving Layer**:
    *   **Purpose**: To provide low-latency predictions from trained models to end-user applications.
    *   **Inference Mechanism**:
        *   **Model Serving Platform**: A dedicated model serving platform (e.g., **Kubernetes-backed microservices**, **TensorFlow Serving**, **SageMaker Endpoints**) would host the deployed models.
        *   **Data Flow**: Real-time input data (e.g., current traffic conditions, time of day) would be ingested via a dedicated low-latency API or Kafka topic, passed to the serving platform, and predictions would be generated.
    *   **Serving Forecasts**:
        *   **API Gateway**: For exposing prediction endpoints securely.
        *   **Caching Layer (e.g., Redis)**: To store frequently requested forecasts and reduce latency.
        *   **Databases**: Processed forecasts could be stored in a low-latency database for retrieval by dashboards or applications.

5.  **Deployment and Orchestration**:
    *   **Cloud Platform**: A major cloud provider like **AWS, GCP, or Azure** would be chosen for its comprehensive suite of services, scalability, and global reach.
    *   **Orchestration**: **Kubernetes (K8s)** would be the core container orchestration tool. All components (Kafka brokers, Spark workers, model serving microservices, databases) would run as containers managed by Kubernetes.
    *   **CI/CD**: **Jenkins, GitLab CI/CD, or GitHub Actions** for automated build, test, and deployment pipelines.

6.  **Scalability, Reliability, and Monitoring**:
    *   **Scalability**: Achieved through:
        *   **Distributed Technologies**: Kafka, Spark, and distributed databases are inherently scalable horizontally.
        *   **Containerization and Orchestration**: Kubernetes allows for easy scaling of individual services up or down based on demand.
        *   **Cloud-Native Services**: Utilizing managed services (e.g., managed Kafka, managed Spark) from cloud providers for automated scaling.
    *   **Reliability**: Ensured by:
        *   **Redundancy and Fault Tolerance**: Kafka's replication, Spark's fault-tolerant processing, database replication, and Kubernetes' self-healing capabilities.
        *   **Disaster Recovery**: Cross-region deployment and regular backups of data and models.
    *   **Monitoring and Alerting**:
        *   **System Metrics**: **Prometheus** for collecting infrastructure and application metrics, paired with **Grafana** for dashboarding and visualization.
        *   **Logging**: Centralized logging with **Elasticsearch, Logstash, Kibana (ELK Stack)** or cloud-native logging services.
        *   **Model Performance Monitoring**: Custom dashboards and alerts for model drift, prediction errors, and data quality issues, integrated with tools like MLflow.
        *   **Alerting**: Integration with communication platforms (e.g., PagerDuty, Slack) for critical alerts.

## Optimization of Public Transportation Services

### Subtask:
Based on the forecasts, discuss strategies and mechanisms for optimizing public transportation services, including route planning, fleet management, and resource allocation to meet predicted demand efficiently.


## Summary

### Data Analysis Key Findings

* **Data Loading and Cleaning**: All five county datasets (Kisumu, Mombasa, Nairobi, Nakuru, Uasin Gishu) were successfully loaded and cleaned. Multi-row headers were parsed, numeric values standardized, and a county identifier added. Each dataset spans **12 years (2013–2024)**, providing 60 total observations.

* **Data Consolidation**: The five DataFrames were combined into a single `df_combined` DataFrame with 60 rows and 10 clean columns. Column name variations across counties were standardized during the cleaning step.

* **Exploratory Data Analysis**:
    * Nairobi dominates road transport volume (~32% of national), followed by Mombasa (~14%) and Nakuru (~9%).
    * Motorcycle registrations are proportionally highest in Kisumu (boda-boda culture) despite being a smaller city.
    * COVID-19 caused a sharp decline in 2020 across all counties, with Mombasa and Nairobi hit hardest.
    * Strong positive correlation between Road Transport expenditure and Year, reflecting economic growth.

* **Feature Engineering**: Created 16 lag features (lag-1 and lag-2 for each transport category), year-over-year growth rates, time trend index, and county encoding — yielding 20 features for model training.

* **Model Selection and Training**: Three models were trained — Random Forest, Gradient Boosting, and Linear Regression — using a time-based train/test split (2013–2021 for training, 2022–2024 for testing).

* **Model Evaluation**: Models were evaluated on MAE, RMSE, R², and MAPE. The ensemble models (Random Forest and Gradient Boosting) significantly outperformed Linear Regression.

* **Forecasting**: The best model was used to project Road Transport demand for 2025–2027, showing continued growth across all counties with Nairobi maintaining the highest absolute demand.

* **Distributed System Architecture**: A conceptual architecture using Apache Kafka (ingestion), Apache Spark (processing), MLflow (model management), and FastAPI (serving) was designed for real-time transportation demand forecasting.

* **Optimization Strategies**: Route planning, dynamic fleet allocation, and data-driven resource management were proposed based on forecasted demand patterns.